In [1]:
from pyspark.sql import SparkSession

basic spark session

In [1]:
spark = SparkSession \
    .builder \
    .appName("day8") \
    .master("spark://spark-master:7077") \
    .enableHiveSupport() \
    .getOrCreate()

configured spark session

In [8]:
spark =    SparkSession.builder \
    .appName("day8") \
    .master("spark://spark-master:7077") \
    .config("spark.hadoop.fs.s3a.endpoint",           "http://minio:9000") \
    .config("spark.hadoop.fs.s3a.access.key",         "minioadmin") \
    .config("spark.hadoop.fs.s3a.secret.key",         "minioadmin") \
    .config("spark.hadoop.fs.s3a.path.style.access",  "true") \
    .config("spark.hadoop.fs.s3a.impl",               "org.apache.hadoop.fs.s3a.S3AFileSystem") \
    .config("spark.sql.catalogImplementation",        "hive") \
    .config("hive.metastore.uris",                    "thrift://metastore:9083") \
    .config("spark.eventLog.enabled",                 "true") \
    .config("spark.eventLog.dir",                     "s3a://spark-event/") \
    .config("spark.sql.shuffle.partitions",           str(4)) \
    .config("spark.sql.adaptive.enabled",             "true") \
    .config("spark.sql.adaptive.coalescePartitions.enabled", "true") \
    .enableHiveSupport() \
    .getOrCreate()


── Cluster ─────────────────────────────────────────────────────────    
.master("spark://spark-master:7077")    

── MinIO (S3) ───────────────────────────────────────────────────────     
.config("spark.hadoop.fs.s3a.endpoint",           "http://minio:9000")    
.config("spark.hadoop.fs.s3a.access.key",         "minioadmin")    
.config("spark.hadoop.fs.s3a.secret.key",         "minioadmin")    
.config("spark.hadoop.fs.s3a.path.style.access",  "true")    
.config("spark.hadoop.fs.s3a.impl",               "org.apache.hadoop.fs.s3a.S3AFileSystem")   

── Hive Metastore ───────────────────────────────────────────────────     
.config("spark.sql.catalogImplementation",        "hive")    
.config("hive.metastore.uris",                    "thrift://metastore:9083")    

── History Server event log ─────────────────────────────────────────     
- Driver writes events here during execution.     
- After spark.stop(), History Server at :18080 reads them back.      
.config("spark.eventLog.enabled",                 "true")     
.config("spark.eventLog.dir",                     "s3a://spark-event/")     

── Local dev tuning ─────────────────────────────────────────────────     
.config("spark.sql.shuffle.partitions",           str(4))    
.config("spark.sql.adaptive.enabled",             "true")     
.config("spark.sql.adaptive.coalescePartitions.enabled", "true")    

In [9]:
spark.sparkContext.setLogLevel("WARN")

data handling

In [22]:
from pyspark.sql import functions as F

In [11]:
# basic

print("=" * 60)
print("Spark version        :", spark.version)
print("Master               :", spark.sparkContext.master)
print("Default parallelism  :", spark.sparkContext.defaultParallelism)
print("App ID               :", spark.sparkContext.applicationId)
print("=" * 60)

Spark version        : 3.5.3
Master               : spark://spark-master:7077
Default parallelism  : 2
App ID               : app-20260411083702-0000


In [12]:
# .option("header", "true") - tell first row is header (csv)
# .option("inferSchema", "true") - force spark to detect data type automatically
# .csv("s3a://raw-data/employees.csv") - read csv
# .option("mergeSchema", "true") - when data schema between files are slightly different

df = spark.read \
    .format("parquet").load("s3a://raw-data/yellow_tripdata_2025*") # read.format.load for when format is dynamic

"""
# some specific months
spark.read.parquet("s3a://raw-data/yellow_tripdata_2025-0{1,2,3}.parquet")
# read folder 
spark.read.parquet("s3a://raw-data/taxi/")
# read multiple files
spark.read.parquet(
    "s3a://raw-data/yellow_tripdata_2025-01.parquet",
    "s3a://raw-data/yellow_tripdata_2025-02.parquet"
)
# read specific files in multiple folder
spark.read.parquet("s3a://raw-data/taxi/*/yellow_*.parquet")
"""

'\n# some specific months\nspark.read.parquet("s3a://raw-data/yellow_tripdata_2025-0{1,2,3}.parquet")\n# read folder \nspark.read.parquet("s3a://raw-data/taxi/")\n# read multiple files\nspark.read.parquet(\n    "s3a://raw-data/yellow_tripdata_2025-01.parquet",\n    "s3a://raw-data/yellow_tripdata_2025-02.parquet"\n)\n# read specific files in multiple folder\nspark.read.parquet("s3a://raw-data/taxi/*/yellow_*.parquet")\n'

In [17]:
# inspect 
print("\n── Schema ─────────────────────────────────────────────")
df.printSchema()

print("\n── First 5 rows ───────────────────────────────────────")
df.show(5, truncate=False)

print("\n── Other ──────────────────────────────────────────────")
print(f"\nTotal rows : {df.count()}")
print(f"Partitions : {df.rdd.getNumPartitions()}") # 3 files for 3 months -> 3 partitions 


── Schema ─────────────────────────────────────────────
root
 |-- VendorID: integer (nullable = true)
 |-- tpep_pickup_datetime: timestamp_ntz (nullable = true)
 |-- tpep_dropoff_datetime: timestamp_ntz (nullable = true)
 |-- passenger_count: long (nullable = true)
 |-- trip_distance: double (nullable = true)
 |-- RatecodeID: long (nullable = true)
 |-- store_and_fwd_flag: string (nullable = true)
 |-- PULocationID: integer (nullable = true)
 |-- DOLocationID: integer (nullable = true)
 |-- payment_type: long (nullable = true)
 |-- fare_amount: double (nullable = true)
 |-- extra: double (nullable = true)
 |-- mta_tax: double (nullable = true)
 |-- tip_amount: double (nullable = true)
 |-- tolls_amount: double (nullable = true)
 |-- improvement_surcharge: double (nullable = true)
 |-- total_amount: double (nullable = true)
 |-- congestion_surcharge: double (nullable = true)
 |-- Airport_fee: double (nullable = true)
 |-- cbd_congestion_fee: double (nullable = true)


── First 5 rows ─

In [32]:
# ── HIVE CHECK ────────────────────────────────────────────────────────────────
# local_db is auto-created by init-hive-db container on first startup
print("\n── Hive databases (local_db should appear) ────────────────────────────")
spark.sql("SHOW DATABASES").show()


── Hive databases (local_db should appear) ────────────────────────────
+---------+
|namespace|
+---------+
|  default|
| local_db|
+---------+



In [20]:
# lazy

high_fare = df.filter(F.col("fare_amount") > 15) \
            .filter(F.col("store_and_fwd_flag") == "N")

# action
count_high_fare = high_fare.count()
print(f"\nfare amount > 15: {count_high_fare}")            


fare amount > 15: 3272317


In [27]:
# trip time > 20 minutes
print(df.withColumn("diff_seconds", F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")) \
    .filter(F.col("diff_seconds") > 1200).count())

2504982


In [29]:
df.withColumn("diff_sec", F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime")).show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|diff_sec|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------+
|       1| 2025-03-01 00:17:16|  2025-03-01 00:25:52|              1|          0.9|         1|         

In [31]:
df.withColumn("diff_sec", col("tpep_dropoff_datetime") - col("tpep_pickup_datetime")).show(5)

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------------------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|            diff_sec|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------------------+
|       1| 2025-03-01 00:17:16|  2025-03-01 00:25:52|              

In [36]:
df = df.withColumn("diff_sec", F.unix_timestamp("tpep_dropoff_datetime") - F.unix_timestamp("tpep_pickup_datetime"))
df_group = df.withColumn("trip_type", F.when( F.col("diff_sec") <= 900, "short").when( F.col("diff_sec") <=1800, "medium").otherwise("long"))
df_group.show(5)
                   

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------+---------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|Airport_fee|cbd_congestion_fee|diff_sec|trip_type|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+------------------+--------+---------+
|       1| 2025-03-01 00:17:16|  2025-03-01 00:25:52|              1|    

In [37]:
print("\n── Explain: groupBy (look for Exchange + double HashAggregate) ─────────")
df_group.groupBy("trip_type").count().explain()
 
print("\n── Extended plan (logical + physical) ────────────────────────────────")
df_group.groupBy("trip_type").count().explain(extended=True)


── Explain: groupBy (look for Exchange + double HashAggregate) ─────────
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[trip_type#1121], functions=[count(1)])
   +- Exchange hashpartitioning(trip_type#1121, 4), ENSURE_REQUIREMENTS, [plan_id=431]
      +- HashAggregate(keys=[trip_type#1121], functions=[partial_count(1)])
         +- Project [CASE WHEN (diff_sec#1099L <= 900) THEN short WHEN (diff_sec#1099L <= 1800) THEN medium ELSE long END AS trip_type#1121]
            +- Project [(unix_timestamp(tpep_dropoff_datetime#2, yyyy-MM-dd HH:mm:ss, Some(Etc/UTC), false) - unix_timestamp(tpep_pickup_datetime#1, yyyy-MM-dd HH:mm:ss, Some(Etc/UTC), false)) AS diff_sec#1099L]
               +- FileScan parquet [tpep_pickup_datetime#1,tpep_dropoff_datetime#2] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(3 paths)[s3a://raw-data/yellow_tripdata_2025-01.parquet, s3a://raw-data/yellow_..., PartitionFilters: [], PushedFilters: [], ReadSc

**add worker node with this command**
```docker compose up -d --scale spark-worker=2```

- if partition < worker, then some workers will be idle

In [40]:
print(df_group.repartition(3).count())
print(df_group.repartition(6).count())

11198026
11198026


In [42]:
print("A — building filter plan")
filtered = df_group.filter(F.col("total_amount") > 20)
print("B — building groupBy plan")
grouped = filtered.groupBy("trip_type").count()
print("C — calling show() — THIS triggers execution")
grouped.show()

A — building filter plan
B — building groupBy plan
C — calling show() — THIS triggers execution
+---------+-------+
|trip_type|  count|
+---------+-------+
|    short|1915911|
|     long| 918192|
|   medium|2926821|
+---------+-------+



In [43]:
df_group.groupBy("trip_type").agg(F.avg("total_amount")).explain()

== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- HashAggregate(keys=[trip_type#1121], functions=[avg(total_amount#16)])
   +- Exchange hashpartitioning(trip_type#1121, 4), ENSURE_REQUIREMENTS, [plan_id=874]
      +- HashAggregate(keys=[trip_type#1121], functions=[partial_avg(total_amount#16)])
         +- Project [total_amount#16, CASE WHEN (diff_sec#1099L <= 900) THEN short WHEN (diff_sec#1099L <= 1800) THEN medium ELSE long END AS trip_type#1121]
            +- Project [total_amount#16, (unix_timestamp(tpep_dropoff_datetime#2, yyyy-MM-dd HH:mm:ss, Some(Etc/UTC), false) - unix_timestamp(tpep_pickup_datetime#1, yyyy-MM-dd HH:mm:ss, Some(Etc/UTC), false)) AS diff_sec#1099L]
               +- FileScan parquet [tpep_pickup_datetime#1,tpep_dropoff_datetime#2,total_amount#16] Batched: true, DataFilters: [], Format: Parquet, Location: InMemoryFileIndex(3 paths)[s3a://raw-data/yellow_tripdata_2025-01.parquet, s3a://raw-data/yellow_..., PartitionFilters: [], PushedFilters: [], ReadSc

  - FileScan    → read data from source
  - Exchange    → transfer data between partition
  - HashAggregate (appears twice!) → 1st group by within partition, 2nd group by after data transfer

In [44]:
spark.stop()
print("\nDone — check :18080 for the job history")


Done — check :18080 for the job history
